In [ ]:
import joblib
from deepchem.feat import CircularFingerprint
from rdkit import Chem
import pubchempy as pcp
from pathlib import Path

# Load DeepChem model
BASE_DIR = Path(__file__).resolve().parent.parent.parent
featurizer = CircularFingerprint(radius=2, size=1024)
rf_esol = joblib.load(BASE_DIR / 'models/rf_esol_model.pkl')

def predict_solubility(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")
    features = featurizer.featurize([mol])
    prediction = rf_esol.predict(features)
    cid = None
    name = "Custom Molecule"
    try:
        compound = pcp.get_compounds(smiles, 'smiles')[0]
        cid = compound.cid
        name = compound.iupac_name or (compound.synonyms[0] if compound.synonyms else "Custom Molecule")
    except:
        pass
    return {
        "id": str(cid) if cid else "custom",
        "name": name,
        "cid": str(cid) if cid else "N/A",
        "formula": Chem.MolToSmiles(mol) if mol else "",
        "smiles": smiles,
        "molecularWeight": f"{Chem.Descriptors.MolWt(mol):.2f} g/mol" if mol else "N/A",
        "logS": f"{prediction[0]:.2f}",
        "solubility": f"{(10 ** prediction[0] * 1000):.2f} g/L (water)",
        "ecfp": "Generated via CircularFingerprint",
    }